# 01 — Kiểm thử source code và dữ liệu

Notebook này khôi phục dự án từ Google Drive, xác minh hai dataset Excel, import
các module chính và chạy toàn bộ pytest. Chạy **Runtime → Run all**.

In [ ]:
#@title Clone dự án từ GitHub và cài môi trường Colab
from google.colab import drive
drive.mount("/content/drive", force_remount=False)

import os
from pathlib import Path
import shutil
import subprocess
import sys

THU_MUC_COLAB_DRIVE = Path("/content/drive/MyDrive/Genetic_ALO_Colab")
THU_MUC_COLAB_DRIVE.mkdir(parents=True, exist_ok=True)
THU_MUC_DU_AN = Path("/content/genetic-alo")
THU_MUC_KET_QUA_DRIVE = THU_MUC_COLAB_DRIVE / "latest_outputs"
REPOSITORY_URL = "https://github.com/duktrung05/genetic-alo.git"

shutil.rmtree(THU_MUC_DU_AN, ignore_errors=True)
subprocess.run(
    [
        "git", "clone", "--depth", "1", "--branch", "main",
        REPOSITORY_URL, str(THU_MUC_DU_AN),
    ],
    check=True,
)

# Khôi phục output mới nhất từ Drive nếu notebook trước đã tạo kết quả.
if THU_MUC_KET_QUA_DRIVE.is_dir():
    shutil.copytree(
        THU_MUC_KET_QUA_DRIVE,
        THU_MUC_DU_AN / "outputs",
        dirs_exist_ok=True,
    )

subprocess.run(
    [
        sys.executable, "-m", "pip", "install", "--quiet",
        "--disable-pip-version-check", "-r",
        str(THU_MUC_DU_AN / "requirements.txt"),
    ],
    check=True,
)

os.chdir(THU_MUC_DU_AN)
if str(THU_MUC_DU_AN) not in sys.path:
    sys.path.insert(0, str(THU_MUC_DU_AN))

def dong_bo_ket_qua() -> Path:
    """Copy all current outputs to Drive so another notebook can reuse them."""
    THU_MUC_KET_QUA_DRIVE.mkdir(parents=True, exist_ok=True)
    shutil.copytree(
        THU_MUC_DU_AN / "outputs",
        THU_MUC_KET_QUA_DRIVE,
        dirs_exist_ok=True,
    )
    return THU_MUC_KET_QUA_DRIVE

print(f"✅ Đã clone dự án tại: {THU_MUC_DU_AN}")
print(f"✅ Python: {sys.version.split()[0]}")
subprocess.run(["git", "log", "-1", "--oneline"], cwd=THU_MUC_DU_AN, check=True)
print("✅ Dataset Excel nằm trong data/instances.")

In [ ]:
#@title Kiểm tra cấu trúc và tính hợp lệ của dataset
from dataset import DatasetValidator, ExcelDatasetLoader

cac_tep_bat_buoc = [
    THU_MUC_DU_AN / "main.py",
    THU_MUC_DU_AN / "main_benchmark.py",
    THU_MUC_DU_AN / "ui_app.py",
    THU_MUC_DU_AN / "data/instances/instance_easy.xlsx",
    THU_MUC_DU_AN / "data/instances/instance_medium.xlsx",
]
tep_thieu = [str(path) for path in cac_tep_bat_buoc if not path.is_file()]
if tep_thieu:
    raise FileNotFoundError("Thiếu file bắt buộc:\n- " + "\n- ".join(tep_thieu))

for ten in ("easy", "medium"):
    path = THU_MUC_DU_AN / f"data/instances/instance_{ten}.xlsx"
    dataset = ExcelDatasetLoader.load_and_validate(str(path))
    report = DatasetValidator.validate_report(dataset)
    if not report["valid"]:
        raise ValueError(f"Dataset {ten} không hợp lệ: {report['errors']}")
    print(f"✅ Dataset {ten.upper()} hợp lệ: {path.name}")

In [ ]:
#@title Xem trực tiếp dữ liệu Excel
import pandas as pd
from IPython.display import display

XEM_DATASET = "easy" #@param ["easy", "medium"]
SO_DONG_MOI_SHEET = 5 #@param {type:"integer"}

duong_dan_excel = THU_MUC_DU_AN / f"data/instances/instance_{XEM_DATASET}.xlsx"
tep_excel = pd.ExcelFile(duong_dan_excel)
print(f"Dataset thật: {duong_dan_excel}")
print("Các sheet:", tep_excel.sheet_names)

for ten_sheet in tep_excel.sheet_names:
    print(f"\n--- {ten_sheet} ---")
    display(pd.read_excel(duong_dan_excel, sheet_name=ten_sheet).head(SO_DONG_MOI_SHEET))

In [ ]:
#@title Chạy toàn bộ pytest
CHAY_TOAN_BO_TEST = True #@param {type:"boolean"}

if CHAY_TOAN_BO_TEST:
    subprocess.run(
        [sys.executable, "-m", "pytest", "-q"],
        cwd=THU_MUC_DU_AN,
        check=True,
    )
    print("✅ Toàn bộ kiểm thử đã pass.")
else:
    print("ℹ️ Đã bỏ qua pytest.")